# 9장 실습 — 통행시간 예측 모델

배차 알고리즘은 "어느 차가 가장 빨리 오는가"를 알아야 합니다.
그 값을 매번 다익스트라로 구하면 시뮬레이션이 느려집니다.
그래서 좌표와 시각만으로 소요시간을 예측하는 모델을 만듭니다. 교재 9장에 대응합니다.

실습 전에 `scikit-learn`과 `LightGBM`을 설치합니다.

```bash
pip install -r requirements-heavy.txt
```

In [ ]:
import sys
from pathlib import Path

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "smartmob").is_dir())
sys.path[:0] = [str(ROOT), str(ROOT / "labs")]

from lab import banner, expect, todo
from smartmob.viz import use_korean_font

use_korean_font()

## 1. 정답을 만듭니다 (교재 9.1)

학습에 쓸 정답은 3장의 최단경로입니다.
무작위 O-D 쌍을 뽑아 시간대별 그래프에서 최단경로 소요시간을 구합니다.

교재는 저장해 둔 2만 건을 씁니다. 여기서는 실행 시간을 줄이기 위해 4천 건을 새로 계산합니다.
실행 시간은 컴퓨터에 따라 달라집니다. 반복해 사용할 때는 결과를 parquet 파일로 저장합니다.

In [ ]:
import time

from smartmob.teaching.eta import FEATURES, TARGET, build_dataset

t0 = time.perf_counter()
df = build_dataset("hanam", n=4000, seed=0)
print(f"{len(df):,}건 만드는 데 {time.perf_counter() - t0:.1f}초")

print("특징:", FEATURES)
print("정답:", TARGET)
df.head(3).round(3)

입력 특징은 직선거리, 시각, 방위각, 좌표이며 라우팅 결과를 포함하지 않습니다.

방위각을 `sin` 과 `cos` 두 개로 나눈 이유는 각도가 원 위의 값이기 때문입니다.
359도와 1도는 붙어 있는데 숫자로는 358만큼 떨어져 있습니다.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    df[FEATURES], df[TARGET], test_size=0.2, random_state=42
)
print(f"학습 {len(X_train):,}건, 검증 {len(X_test):,}건")

## 2. 기준선 (교재 9.2)

직선거리를 학습 자료에서 구한 평균 속도로 나눈 값을 기준선으로 사용합니다. 이후 모델의 MAE를 이 값과 비교합니다.

In [ ]:
import numpy as np
from sklearn.metrics import mean_absolute_error

avg_speed_kmh = X_train["straight_km"].sum() / (y_train.sum() / 60)
baseline_pred = X_test["straight_km"] / avg_speed_kmh * 60

baseline_name = f"기준선 (직선거리 ÷ {avg_speed_kmh:.1f}km/h)"
scores = {baseline_name: mean_absolute_error(y_test, baseline_pred)}
print(f"평균 속도 {avg_speed_kmh:.1f}km/h")
print(f"평균 절대오차 {scores[baseline_name]:.2f}분")

## 3. 선형회귀 (교재 9.3)

In [ ]:
from sklearn.linear_model import LinearRegression

linear = LinearRegression().fit(X_train, y_train)
scores["선형회귀"] = mean_absolute_error(y_test, linear.predict(X_test))
print(f"평균 절대오차 {scores['선형회귀']:.2f}분")

for name, coef in sorted(zip(FEATURES, linear.coef_), key=lambda x: -abs(x[1]))[:4]:
    print(f"  {name:14s} {coef:+8.3f}")

## 4. 그래디언트 부스팅 (교재 9.4)

In [ ]:
import lightgbm as lgb

gbm = lgb.LGBMRegressor(
    n_estimators=300, learning_rate=0.05, num_leaves=31,
    random_state=42, verbose=-1,
)
gbm.fit(X_train, y_train)
scores["LightGBM"] = mean_absolute_error(y_test, gbm.predict(X_test))
print(f"평균 절대오차 {scores['LightGBM']:.2f}분")

In [ ]:
import pandas as pd

board = pd.DataFrame({"평균절대오차_분": scores}).round(3)
board["기준선 대비"] = (board["평균절대오차_분"] / scores[baseline_name]).round(2)
board

## 5. 특징별 분할 횟수 (교재 9.5)

In [ ]:
import matplotlib.pyplot as plt

importance = pd.Series(gbm.feature_importances_, index=FEATURES).sort_values()

fig, ax = plt.subplots(figsize=(7, 4))
ax.barh(importance.index, importance.values, color="#4C6EF5")
ax.set_xlabel("분할 횟수")
ax.set_title("LightGBM 이 무엇을 보았는가")
plt.tight_layout();

## 6. 어디서 틀리는가 (교재 9.7)

평균 오차와 함께 실제 소요시간 및 직선거리에 따른 오차 분포를 확인합니다.

In [ ]:
pred = gbm.predict(X_test)
error = pred - y_test

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].scatter(y_test, pred, s=6, alpha=0.3, color="#4C6EF5")
lims = [0, max(y_test.max(), pred.max())]
axes[0].plot(lims, lims, "k--", lw=1)
axes[0].set_xlabel("실제 (분)")
axes[0].set_ylabel("예측 (분)")
axes[0].set_title("예측 대 실제")

axes[1].scatter(X_test["straight_km"], error, s=6, alpha=0.3, color="crimson")
axes[1].axhline(0, color="black", lw=1)
axes[1].set_xlabel("직선거리 (km)")
axes[1].set_ylabel("오차 (분)")
axes[1].set_title("거리에 따른 오차")
plt.tight_layout();

## 7. 배차에 넣어 보기

이 모델이 실제로 쓰이는 자리는 배차의 비용행렬입니다.
도로망 라우팅으로 만든 행렬과 얼마나 비슷한지 봅니다.

In [ ]:
from smartmob.data import load_road_graph
from smartmob.teaching.dispatch import (
    cost_matrix,
    cost_matrix_from_model,
    cost_matrix_from_router,
)

G = load_road_graph(
    "hanam", modes=("drive",), speed_column="weekday_pm_peak_p50"
)
passengers = [(37.539, 127.215), (37.545, 127.200), (37.552, 127.190)]
vehicles = [(37.540, 127.210), (37.560, 127.195), (37.535, 127.225)]

truth = cost_matrix_from_router(passengers, vehicles, G)
straight = cost_matrix(passengers, vehicles)
model = cost_matrix_from_model(passengers, vehicles, gbm.predict, hour=18)

banner("비용행렬 세 가지의 평균 절대오차 (도로망 기준)")
print(f"직선거리 ÷ 25km/h  {np.abs(straight - truth).mean():.2f}분")
print(f"LightGBM          {np.abs(model - truth).mean():.2f}분")

이 3×3 예제는 18시 속도를 적용한 도로망 비용을 기준으로 두 근사 방법의 MAE를 출력합니다. 검증 자료 전체의 평균 성능이 특정 배차 후보에서도 항상 우수한 것은 아닙니다.

## 8. 모델 조건 바꾸기

### 8.1 특징 하나 추가하기

`FEATURES` 에 없는 특징을 하나 만들어 넣고 오차가 줄어드는지 봅니다.
후보는 여럿입니다. 출발지와 목적지의 위도 차이, 경도 차이, 도심으로부터의 거리 같은 것입니다.

실제 사용할 때 라우팅 없이 계산할 수 있는 특징만 사용합니다.

In [ ]:
my_feature_name = None      # 추가한 특징의 이름
my_feature_mae = None       # 그때의 평균 절대오차 (분)

banner("빈칸 8.1")
todo("추가한 특징", my_feature_name)
todo("그때의 오차", my_feature_mae, fmt=lambda v: f"{v:.2f}분")

### 8.2 학습 데이터를 늘리면

`build_dataset`의 `n`을 2배, 4배로 늘려 가며 데이터 증가량 대비 MAE 감소 폭을 계산합니다.
데이터를 늘리는 것과 모델을 바꾸는 것 중 어느 쪽이 나은지 두 줄로 적습니다.

In [ ]:
saturation_n = None     # 오차가 더 이상 줄지 않기 시작하는 데이터 크기

banner("빈칸 8.2")
todo("포화 데이터 크기", saturation_n)

### 8.3 가장 크게 틀린 통행

오차가 가장 큰 통행 다섯 건을 찾아 출발지와 목적지를 지도에 표시합니다. 각 통행의 실제 경로와 직선거리 대비 도로거리를 함께 확인합니다.

In [ ]:
worst_error_min = None      # 가장 큰 절대오차 (분)

banner("빈칸 8.3")
todo("가장 큰 오차", worst_error_min, fmt=lambda v: f"{v:.1f}분")

## 정리

- 학습 목표값은 3장의 최단경로로 계산합니다
- 직선거리와 학습 자료의 평균 속도로 기준선을 만듭니다
- 각도는 `sin` 과 `cos` 두 값으로 나눠 넣습니다
- 평균 오차와 함께 시간대·거리 구간별 오차를 확인합니다
- 10장 실습에서는 이 예측값으로 배차 비용행렬을 만듭니다